# 01. Multi-Agent 패턴 개요

> 단일 에이전트로 풀기 힘든 작업은 여러 에이전트를 협력시켜 풀어요. Subagents · Handoffs · Router · Skills · Custom Workflow 5가지 패턴의 차이를 한눈에 비교해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. 멀티에이전트(Multi-Agent) 시스템이 필요한 이유를 설명할 수 있어요
2. 5가지 핵심 멀티에이전트 패턴(Subagents, Handoffs, Router, Skills, Custom Workflow)을 비교하고 구분할 수 있어요
3. 각 패턴의 모델 호출 수, 토큰 비용, 적합한 시나리오를 파악할 수 있어요
4. 실제 문제 상황에 맞는 패턴을 선택하고 간단한 구현 예제를 작성할 수 있어요

## 사전 지식

- Part 07 `02-Long-Term-Memory.ipynb`에서 배운 Store API와 세션 간 공유
- LangGraph StateGraph, 노드(Node), 엣지(Edge) 기초
- `create_agent` 함수와 LangChain V1 에이전트 기초
- Tool 정의와 바인딩 방법

## 왜 멀티에이전트인가?

단일 에이전트로 모든 것을 처리하면 어떤 문제가 생길까요?

| 문제 | 설명 | 영향 |
|------|------|------|
| **도구 과부하** | 한 에이전트에 도구가 너무 많아지면 잘못된 도구를 선택함 | 정확도 저하 |
| **컨텍스트 오버플로우** | 대화가 길어질수록 중요한 정보를 잊어버림 | 일관성 깨짐 |
| **전문화 부재** | 하나의 에이전트가 모든 분야를 다루면 깊이가 없음 | 품질 저하 |
| **확장성 한계** | 기능을 추가할 때마다 프롬프트가 복잡해짐 | 유지보수 어려움 |

멀티에이전트 시스템은 이 문제들을 **분업(Specialization)**과 **조율(Orchestration)**로 해결해요.

> 💡 **핵심 정리**: 멀티에이전트 시스템은 **회사 조직**과 같아요. 모든 일을 혼자 하는 1인 기업 vs 전문가 팀 시스템의 차이를 생각해보세요. 1인 기업(단일 에이전트)은 단순한 일에 빠르지만, 복잡한 프로젝트에서는 팀(멀티에이전트)이 훨씬 효과적이에요.

> 🔑 **핵심 개념**: 멀티에이전트 시스템의 핵심은 **각 에이전트가 자신의 전문 영역에만 집중**하고, **오케스트레이터(Orchestrator)**가 전체 흐름을 관리한다는 것이에요.

### 5가지 패턴을 한 줄로 요약하면?

| 패턴 | 한 줄 비유 |
|------|-----------|
| **Subagents** | 팀장이 팀원에게 업무를 지시하고 보고받는 구조 |
| **Handoffs** | 공항 체크인 → 보안검색 → 탑승구처럼 고객을 다음 창구로 안내 |
| **Router** | 우체국에서 편지를 지역별로 분류해 동시에 배달하는 구조 |
| **Skills** | 스마트폰에 앱을 필요할 때만 설치하는 것처럼 능력을 동적 로딩 |
| **Custom Workflow** | 위 4가지를 레고 블록처럼 조합하는 맞춤형 설계 |

## 5가지 멀티에이전트 패턴 전체 아키텍처

LangChain V1은 5가지 멀티에이전트 패턴을 제공해요. 각 패턴은 서로 다른 문제를 해결하기 위해 설계되었어요.

```mermaid
flowchart TB
    U(["사용자 요청<br>User Request"])

    subgraph patterns["5가지 멀티에이전트 패턴"]
        P1["1. Subagents<br>도구로 위임"]
        P2["2. Handoffs<br>에이전트 간 인계"]
        P3["3. Router<br>병렬 분기"]
        P4["4. Skills<br>온디맨드 능력 로딩"]
        P5["5. Custom Workflow<br>패턴 조합"]
    end

    A1["전문 에이전트 A"]
    A2["전문 에이전트 B"]
    R(["최종 응답"])

    U --> patterns
    P1 --> A1
    P2 --> A1
    P3 --> A1
    P3 --> A2
    P4 --> A1
    P5 --> A1
    P5 --> A2
    A1 --> R
    A2 --> R

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class U input
    class P1,P2,P3,P4,P5 process
    class A1,A2 storage
    class R output
```

각 패턴의 핵심 차이를 먼저 표로 정리해볼게요:

| 패턴 | 제어 흐름 | 모델 호출 수 | 주요 사용 사례 |
|------|----------|------------|---------------|
| **Subagents** | 중앙 집중식 (도구 호출) | 높음 (N+1) | 작업 오케스트레이션 |
| **Handoffs** | 분산형 (에이전트 인계) | 중간 (N) | 다중 도메인 대화 |
| **Router** | 병렬 분기 | 낮음 (1+parallel) | 입력 분류 후 병렬 처리 |
| **Skills** | 단일 에이전트 확장 | 낮음 (1) | 온디맨드 능력 확장 |
| **Custom Workflow** | 혼합 | 가변 | 복잡한 맞춤형 파이프라인 |

## 환경 설정

In [ ]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
# .env 파일에서 API 키를 불러와요
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# ---------------------------------------------------
# LangSmith 추적 설정 (선택사항)
# ---------------------------------------------------
# 실행 과정을 LangSmith에서 시각적으로 추적해요
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-MultiAgent-Overview"

# LangSmith 추적 설정 완료

## 1. Subagents 패턴 - 도구로서의 에이전트

### 개념

**Subagents 패턴**은 중앙 슈퍼바이저 에이전트가 다른 에이전트를 **도구(tool)**로 호출하는 방식이에요.

마치 팀장이 팀원에게 업무를 지시하고, 결과를 보고받는 구조와 같아요.

```mermaid
flowchart LR
    S(["슈퍼바이저<br>Supervisor"])
    T1["Research Agent<br>@tool"]
    T2["Math Agent<br>@tool"]
    T3["Writer Agent<br>@tool"]

    S -- "도구 호출" --> T1
    S -- "도구 호출" --> T2
    S -- "도구 호출" --> T3
    T1 -- "결과 반환" --> S
    T2 -- "결과 반환" --> S
    T3 -- "결과 반환" --> S

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085

    class S input
    class T1,T2,T3 process
```

**특징:**
- 슈퍼바이저가 **모든 컨텍스트를 유지**해요
- 하위 에이전트는 독립적으로 실행되고 **결과만 반환**해요
- 슈퍼바이저가 어떤 에이전트를 호출할지 **LLM이 결정**해요

> 🔑 **핵심 개념**: Subagents 패턴에서 하위 에이전트는 사용자와 직접 대화하지 않아요. 슈퍼바이저가 중간에서 모든 커뮤니케이션을 처리해요.

> ⚠️ **자주 하는 실수**: 하위 에이전트에 너무 많은 도구를 줘서 각 에이전트가 과부하되는 경우가 있어요. 각 에이전트는 **단일 책임 원칙**을 따라야 해요.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 모델: gpt-4o-mini (비용 효율적)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. Handoffs 패턴 - 에이전트 간 제어권 인계

### 개념

**Handoffs 패턴**은 현재 에이전트가 다른 에이전트로 **제어권(control)**을 넘기는 방식이에요. `Command(update={...})` 객체와 `ToolMessage` 페어링을 사용해요.

> 💡 **핵심 정리**: Handoffs는 **병원 진료 시스템**과 같아요. 접수 창구(Supervisor)에서 증상을 듣고, 내과(Agent A)로 안내하고, 필요하면 영상의학과(Agent B)로 의뢰서와 함께 보내요. 각 단계에서 환자의 차트(State)가 함께 전달돼요.

```mermaid
flowchart LR
    U(["사용자"])
    A1["에이전트 A<br/>(현재 활성)"]
    A2["에이전트 B<br/>(인계 대상)"]
    A3["에이전트 C<br/>(인계 대상)"]

    U -- "메시지" --> A1
    A1 -- "Command(goto=B)" --> A2
    A2 -- "Command(goto=C)" --> A3
    A3 -- "최종 응답" --> U

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class U input
    class A1,A2,A3 process
```

**Subagents vs Handoffs 차이:**

| 특성 | Subagents | Handoffs |
|------|----------|----------|
| 제어 흐름 | 중앙 집중 (팀장이 관리) | 분산 (에이전트끼리 직접 전달) |
| 컨텍스트 | 슈퍼바이저가 유지 | 에이전트가 직접 전달 |
| 사용자 상호작용 | 슈퍼바이저만 | 각 에이전트가 직접 가능 |
| 적합한 상황 | 작업 오케스트레이션 | 다중 도메인 대화 |

> 💡 **실무 팁**: 공식 문서에서는 단일 에이전트 + 미들웨어 조합을 Handoffs 패턴의 권장 구현으로 소개해요. 단순한 케이스에서는 에이전트를 여러 개 만드는 것보다 미들웨어로 제어하는 것이 더 효율적이에요.

In [ ]:
from typing import Annotated
from langchain_core.tools import InjectedToolCallId
from langgraph.prebuilt import InjectedState
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.types import Command

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → supervisor_agent → (billing_agent | tech_agent) → supervisor_agent → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. Router 패턴 - 병렬 분기 처리

### 개념

**Router 패턴**은 입력을 분류(classify)하여 여러 에이전트에게 **병렬로 dispatch**하는 방식이에요. `Send` API의 자세한 문법은 Part 02에서 이미 배웠고, 여기서는 멀티 에이전트 패턴 관점에서만 짧게 복습합니다.

```mermaid
flowchart LR
    I(["입력"])
    R["라우터<br>Router"]
    A1["에이전트 A<br>(경로 1)"]
    A2["에이전트 B<br>(경로 2)"]
    A3["에이전트 C<br>(경로 3)"]
    O(["결과 집계"])

    I --> R
    R -- "Send(A, ...)" --> A1
    R -- "Send(B, ...)" --> A2
    R -- "Send(C, ...)" --> A3
    A1 --> O
    A2 --> O
    A3 --> O

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class I input
    class R,A1,A2,A3 process
    class O output
```

**Router 패턴의 강점:**
- 여러 에이전트가 **동시에(병렬로)** 작업 처리
- 라우팅 로직이 비교적 명확해서 비용과 지연시간을 예측하기 쉬움
- 같은 질문을 여러 데이터 소스나 전문 역할에 나누어 보낼 때 효과적

> 🔁 **복습 연결**: `Send`는 `02_LangGraph_Basics/04-StateGraph-Basics.ipynb`의 동적 병렬 처리에서 배운 기능입니다. 이 장에서는 `Send` 자체보다 “어떤 멀티 에이전트 문제에 Router가 맞는가?”에 집중하세요.


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → translate (병렬 분기) → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. Skills 패턴 - 온디맨드 능력 로딩

### 개념

**Skills 패턴**은 단일 에이전트가 필요에 따라 **능력(skill)**을 동적으로 로드하는 방식이에요. 에이전트를 여러 개 만드는 대신, 하나의 에이전트가 상황에 맞는 능력을 꺼내 써요.

```mermaid
flowchart LR
    U(["사용자"])
    A["단일 에이전트"]
    SM["SkillMiddleware"]
    S1["SQL 스킬"]
    S2["코드 리뷰 스킬"]
    S3["데이터 분석 스킬"]

    U -- "SQL 질문" --> A
    A -- "load_skill('sql')" --> SM
    SM -- "SQL 지식 로드" --> S1
    S1 -- "스킬 주입" --> A
    A -- "응답" --> U

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class U input
    class A,SM process
    class S1,S2,S3 storage
```

**Skills 패턴의 특징:**
- 에이전트가 **1개**뿐이므로 관리가 단순해요
- 스킬은 TypedDict로 정의하고 필요할 때 로드해요
- `load_skill` 도구와 `SkillMiddleware`를 활용해요

> 💡 **핵심 정리**: Skills 패턴은 스마트폰 앱처럼 기본 에이전트에 앱을 설치하는 개념이에요. 필요할 때만 앱을 실행하고, 나머지는 배경에서 대기해요. 에이전트 수를 줄이면서 유연성은 높일 수 있어요.

> ⚠️ **자주 하는 실수**: 스킬에 너무 많은 내용을 넣으면 컨텍스트가 오히려 늘어나요. 각 스킬은 **하나의 전문 분야**에 집중해야 해요.

In [ ]:
from typing import TypedDict

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain.tools import tool

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트가 SQL 질문을 감지하고 스스로 sql 스킬을 로드하는지 확인해요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. Custom Workflow 패턴 - 패턴 조합

### 개념

**Custom Workflow 패턴**은 위에서 배운 4가지 패턴을 **자유롭게 조합**하여 맞춤형 멀티에이전트 시스템을 만드는 방식이에요.

```mermaid
flowchart TB
    U(["사용자 요청"])
    R["Router<br>(분류)"]
    S1["Subagent 흐름<br>(Research + Math)"]
    S2["Handoffs 흐름<br>(전문가 인계)"]
    SK["Skills 흐름<br>(온디맨드)"]    
    AGG["결과 집계"]
    O(["최종 응답"])

    U --> R
    R -- "분석 요청" --> S1
    R -- "지원 요청" --> S2
    R -- "전문 지식" --> SK
    S1 --> AGG
    S2 --> AGG
    SK --> AGG
    AGG --> O

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class U input
    class R,S1,S2,SK,AGG process
    class O output
```

**실제 사용 사례:**
- 고객 지원 시스템: Router(문의 분류) + Handoffs(전문팀 인계) + Skills(제품 지식 로드)
- 리서치 에이전트: Subagents(정보 수집) + Router(병렬 검색) + Custom(최종 합성)

> 🔑 **핵심 개념**: Custom Workflow는 '없는 것을 만드는' 것이 아니라, 이미 배운 패턴들을 **적재적소에 조합**하는 것이에요. 각 패턴의 강점을 이해하면 자연스럽게 최적의 조합이 떠올라요.

> 💡 **실무 팁**: Custom Workflow를 설계할 때는 먼저 **플로우차트**를 그려보세요. 어떤 패턴을 어디에 쓸지 시각화하면 구현이 훨씬 쉬워져요.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → classify → (math_flow | knowledge_flow | general_flow) → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 패턴 선택 의사결정 트리

어떤 패턴을 선택해야 할지 고민될 때 이 의사결정 트리를 따라가 보세요.

```mermaid
flowchart TD
    Q1{"에이전트 여러 개가<br>필요한가?"}
    Q2{"병렬 처리가<br>필요한가?"}
    Q3{"사용자와 직접<br>대화해야 하나?"}
    Q4{"다단계<br>추론이 필요한가?"}

    A1["Skills 패턴<br>단일 에이전트 + 능력 로딩"]
    A2["Router 패턴<br>Send API 병렬 분배"]
    A3["Handoffs 패턴<br>에이전트 간 제어권 인계"]
    A4["Subagents 패턴<br>중앙 집중 오케스트레이션"]
    A5["Custom Workflow<br>패턴 조합"]

    Q1 -- No --> A1
    Q1 -- Yes --> Q2
    Q2 -- Yes --> A2
    Q2 -- No --> Q3
    Q3 -- Yes --> A3
    Q3 -- No --> Q4
    Q4 -- Yes --> A4
    Q4 -- No --> A5

    classDef question fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef answer fill:#d4edda,stroke:#28a745,color:#155724
    class Q1,Q2,Q3,Q4 question
    class A1,A2,A3,A4,A5 answer
```

> 💡 **실무 팁**: 처음에는 가장 단순한 패턴(Skills)으로 시작하세요. 요구사항이 복잡해지면 점진적으로 더 강력한 패턴으로 전환하는 것이 안전해요.

## 6. 패턴 비교 및 선택 가이드

### Capability Matrix (능력 매트릭스)

각 패턴이 어떤 특성을 가지는지 비교해볼게요:

| 특성 | Subagents | Handoffs | Router | Skills | Custom |
|------|:---------:|:--------:|:------:|:------:|:------:|
| 분산 개발 용이성 | ★★★ | ★★★ | ★★ | ★★ | ★★★ |
| 병렬화 지원 | ★ | ★★ | ★★★ | ★ | ★★★ |
| Multi-hop (다단계 추론) | ★★★ | ★★★ | ★ | ★★ | ★★★ |
| 사용자 상호작용 | ★ | ★★★ | ★★ | ★★ | ★★★ |
| 구현 복잡도 | 중간 | 높음 | 낮음 | 낮음 | 높음 |
| 토큰 비용 | 높음 | 중간 | 낮음 | 낮음 | 가변 |

### Optimization Quick Reference

| 시나리오 | 권장 패턴 | 이유 |
|---------|----------|------|
| 독립적인 병렬 작업 | Router | Send API로 최소 모델 호출 |
| 순차적 전문가 인계 | Handoffs | 에이전트가 직접 사용자와 대화 |
| 중앙 집중식 오케스트레이션 | Subagents | 슈퍼바이저가 전체 컨텍스트 유지 |
| 단일 에이전트 능력 확장 | Skills | 에이전트 수 최소화 |
| 복합 비즈니스 로직 | Custom | 여러 패턴의 장점 조합 |

> 💡 **핵심 정리**: "어떤 패턴이 '최고'인지는 없어요. 문제 특성에 맞는 패턴을 선택하는 것이 핵심이에요. 처음에는 가장 단순한 패턴부터 시작해보세요."

> ⚠️ **자주 하는 실수**: 모든 문제에 Custom Workflow를 쓰려는 경향이 있어요. 단순한 문제는 단순한 패턴으로도 충분히 해결할 수 있어요. 오버엔지니어링을 피하세요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 실습: 나만의 패턴 선택해보기

아래 TODO 블록을 수정해서 직접 실험해보세요!

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 자신의 시나리오에 맞는 패턴을 선택해보세요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **Subagents 패턴**: 슈퍼바이저가 하위 에이전트를 `@tool`로 래핑하여 도구처럼 호출해요. 중앙 집중식 제어, 컨텍스트 유지에 강점
- **Handoffs 패턴**: `Command(goto=agent_name)` + `ToolMessage` 페어링으로 에이전트 간 제어권을 넘겨요. 다중 도메인 대화에 적합
- **Router 패턴**: `Send` API로 여러 에이전트에 병렬 dispatch해요. 모델 호출 최소화, 비용 효율적
- **Skills 패턴**: 단일 에이전트가 `load_skill` 도구로 전문 지식을 동적 로드해요. 에이전트 수 최소화
- **Custom Workflow**: 위 4가지 패턴을 조합하여 복잡한 비즈니스 로직을 구현해요
- **패턴 선택 기준**: 병렬화 필요성, 사용자 상호작용, 다단계 추론, 복잡도를 기준으로 선택해요

## 다음 노트북 예고

다음 `02-Supervisor.ipynb`에서는 **기본 Supervisor 패턴**을 배워요. `langgraph-supervisor` 라이브러리를 사용한 빠른 구현부터, `StateGraph`로 직접 Supervisor를 구현하는 방법까지 단계별로 학습해요.